In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os, glob
import numpy as np
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Mining of massive dataset/preprocessing_output/green_data/green_2018/"
LOOKUP_PATH = "/content/drive/MyDrive/Mining of massive dataset/preprocessing_output/taxi_zone_lookup_grid.csv"

In [3]:
file_list = sorted(glob.glob(os.path.join(DATA_DIR, "*_volume.csv")))
df_all = pd.concat([pd.read_csv(f) for f in file_list], ignore_index=True)

df_all = df_all.rename(columns={"locationid": "LocationID"})
df_lookup = pd.read_csv(LOOKUP_PATH)
df_merge = pd.merge(df_all, df_lookup, on="LocationID", how="left")

In [4]:
print("Merged rows:", len(df_merge))
print(df_merge.head())

Merged rows: 2610314
              time_bin  LocationID  start_volume  end_volume  Grid_X  Grid_Y
0  2009-01-01 10:00:00          93             1           0       7      11
1  2009-01-01 10:00:00         117             0           1       8       3
2  2009-01-01 13:00:00           7             1           0       6      12
3  2009-01-01 13:00:00          93             1           0       7      11
4  2009-01-01 13:00:00         117             0           2       8       3


In [5]:
df_merge["time_bin"] = pd.to_datetime(df_merge["time_bin"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
df_merge = df_merge.dropna(subset=["time_bin", "Grid_X", "Grid_Y"])

df_merge["Grid_X"] = df_merge["Grid_X"].astype(int)
df_merge["Grid_Y"] = df_merge["Grid_Y"].astype(int)

df_merge["start_volume"] = pd.to_numeric(df_merge["start_volume"], errors="coerce").fillna(0).astype(np.float32)
df_merge["end_volume"]   = pd.to_numeric(df_merge["end_volume"], errors="coerce").fillna(0).astype(np.float32)

In [6]:
df_merge = df_merge[(df_merge["time_bin"] >= "2018-01-01") & (df_merge["time_bin"] < "2019-01-01")].copy()

print("After cleaning:", len(df_merge))
print(df_merge[["time_bin","LocationID","Grid_X","Grid_Y","start_volume","end_volume"]].head())

After cleaning: 2609434
     time_bin  LocationID  Grid_X  Grid_Y  start_volume  end_volume
88 2018-01-01           4       4      10           0.0         3.0
89 2018-01-01           7       6      12          51.0        32.0
90 2018-01-01          10       8       8           1.0         0.0
91 2018-01-01          11       4       4           1.0         0.0
92 2018-01-01          17       5       8          11.0        15.0


In [7]:
df_merge = df_merge.sort_values(by="time_bin")
unique_times = sorted(df_merge["time_bin"].unique())
time_to_idx = {t:i for i,t in enumerate(unique_times)}
df_merge["time_idx"] = df_merge["time_bin"].map(time_to_idx).astype(int)

T = len(unique_times)

C, H, W = 3, 10, 20

grid_x = df_merge["Grid_X"].values
grid_y = df_merge["Grid_Y"].values
time_indices = df_merge["time_idx"].values

start_vols = df_merge["start_volume"].values
end_vols = df_merge["end_volume"].values

data = np.zeros((T, C, H, W), dtype=np.float32)

data[time_indices, 0, grid_x, grid_y] = start_vols
data[time_indices, 1, grid_x, grid_y] = end_vols
data[time_indices, 2, grid_x, grid_y] = start_vols - end_vols

In [8]:
print("Tensor shape:", data.shape)
print("Nonzero:", np.count_nonzero(data))

Tensor shape: (17519, 3, 10, 20)
Nonzero: 2442776


In [9]:
np.save(os.path.join(DATA_DIR, "taxi_volume_4d_tensor_final_green.npy"), data)
np.save(os.path.join(DATA_DIR, "time_bins_green.npy"), np.array(unique_times, dtype="datetime64[s]"))